# Deep Agents on NVIDIA Nemotron

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-nvidia/blob/main/cookbook/nemotron_deep_agent_nim.ipynb)

[Deep Agents](https://docs.langchain.com/oss/python/deepagents/overview) (the `deepagents` package) is an agent
harness built on LangChain and LangGraph. On top of a tool-calling loop it adds:

- **Planning:** a `write_todos` tool the agent uses to break down and track work.
- **A filesystem:** `ls`, `read_file`, `write_file`, `edit_file`, `glob`, and `grep` tools over a pluggable backend.
- **Subagents:** a `task` tool that delegates work to specialized agents with their own context.

This notebook runs a Deep Agent on `nvidia/nemotron-3-super-120b-a12b` through `ChatNVIDIA`, against NVIDIA-hosted endpoints or a
self-hosted NIM. The agent works on a local folder with a calculator tool, so no third-party API keys are needed.

`deepagents` requires Python 3.11 or newer.

## Install the packages

In [ ]:
%pip install -qU "deepagents>=0.7,<0.8" "langchain-nvidia-ai-endpoints>=1.4,<2" "langchain>=1.0,<2" "langgraph>=1.0,<2"

## Configure the endpoint: hosted or self-hosted

Every model client in this notebook is built from the next cell.

| Variable | Unset (default) | Set |
| --- | --- | --- |
| `NVIDIA_BASE_URL` | NVIDIA-hosted endpoints on [build.nvidia.com](https://build.nvidia.com/) | Your own NIM, e.g. `http://localhost:8000/v1` |
| `NVIDIA_API_KEY` | Required for hosted endpoints (you are prompted if it is missing) | Optional for most self-hosted NIMs |
| `NVIDIA_CHAT_MODEL` | `nvidia/nemotron-3-super-120b-a12b` | The model your NIM serves (see `GET $NVIDIA_BASE_URL/models`) |

In [ ]:
import getpass
import os

# Unset: NVIDIA-hosted endpoints (build.nvidia.com).
# Set, e.g. "http://localhost:8000/v1": a self-hosted NIM.
NVIDIA_BASE_URL = os.environ.get("NVIDIA_BASE_URL") or None
CHAT_MODEL = os.environ.get("NVIDIA_CHAT_MODEL", "nvidia/nemotron-3-super-120b-a12b")

if NVIDIA_BASE_URL:
    print(f"Using self-hosted NIM at {NVIDIA_BASE_URL}")
else:
    print("Using NVIDIA-hosted endpoints (build.nvidia.com)")
    if not os.environ.get("NVIDIA_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-...): ")

# Shared keyword arguments for every NVIDIA client in this notebook.
# ChatNVIDIA reads NVIDIA_API_KEY from the environment.
NIM_KWARGS = {"base_url": NVIDIA_BASE_URL} if NVIDIA_BASE_URL else {}
print(f"Chat model: {CHAT_MODEL}")

## Create the chat model

Deep Agents needs a model with reliable tool calling. You can also pass a model string such as
`f"nvidia:{CHAT_MODEL}"` to `create_deep_agent`; it is resolved with `init_chat_model`, which reads
`NVIDIA_BASE_URL` and `NVIDIA_API_KEY` from the environment.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(model=CHAT_MODEL, temperature=0, **NIM_KWARGS)

## Prepare a workspace

The agent reads and writes files in a local `deep_agent_workspace/` folder.

In [ ]:
from pathlib import Path

WORKSPACE = Path("deep_agent_workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)
(WORKSPACE / "expenses.csv").write_text(
    """date,category,amount_usd
2025-01-03,travel,412.50
2025-01-07,meals,38.20
2025-01-09,software,99.00
2025-01-14,travel,186.75
2025-01-21,meals,64.10
2025-01-28,software,249.00
"""
)
sorted(p.name for p in WORKSPACE.iterdir())

## Define a tool

In [ ]:
import ast
import operator

from langchain_core.tools import tool

_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _evaluate(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPERATORS:
        left, right = _evaluate(node.left), _evaluate(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("exponent too large")
        return _OPERATORS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPERATORS:
        return _OPERATORS[type(node.op)](_evaluate(node.operand))
    raise ValueError("only numbers and + - * / // % ** ( ) are allowed")


@tool
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression, for example '(12.5 + 7) * 3'."""
    try:
        return str(_evaluate(ast.parse(expression, mode="eval").body))
    except (SyntaxError, ValueError, ZeroDivisionError) as exc:
        return f"Error: {exc}"


calculator.invoke({"expression": "(12.5 + 7) * 3"})

## Create the Deep Agent

`FilesystemBackend(root_dir=..., virtual_mode=True)` exposes the workspace to the agent as `/`, and blocks paths
outside it. The `reviewer` subagent shows how to delegate a focused task with its own instructions.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

reviewer = {
    "name": "reviewer",
    "description": "Reviews a draft report for arithmetic and clarity problems.",
    "system_prompt": (
        "You review short expense reports. Check every total with the calculator "
        "tool and reply with a list of concrete corrections, or 'LGTM'."
    ),
    "tools": [calculator],
}

agent = create_deep_agent(
    model=llm,
    tools=[calculator],
    system_prompt=(
        "You are a careful finance assistant. Plan with write_todos, read input files "
        "from the filesystem, use the calculator for all arithmetic, and save results "
        "with write_file."
    ),
    subagents=[reviewer],
    backend=FilesystemBackend(root_dir=WORKSPACE, virtual_mode=True),
)

## Run the agent

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Read /expenses.csv, total the amounts per category and overall, "
                "and write a short Markdown summary to /report.md. Ask the reviewer "
                "subagent to check it before you finish.",
            }
        ]
    }
)
result["messages"][-1].pretty_print()

Inspect the agent's plan and the file it wrote.

In [ ]:
for todo in result.get("todos", []):
    print(f"[{todo['status']}] {todo['content']}")

report = WORKSPACE / "report.md"
print(report.read_text() if report.exists() else "The agent did not write /report.md")

## Going further with NVIDIA integrations

This repository has two Deep Agents add-ons that work with the agent above:

- [`langchain-nvidia-openshell`](https://github.com/langchain-ai/langchain-nvidia/tree/main/libs/openshell): an `OpenShellSandbox` backend that runs the agent's
  `execute` and file tools inside a policy-enforced [NVIDIA OpenShell](https://github.com/NVIDIA/OpenShell)
  sandbox while the agent stays on the host. Swap it in for `FilesystemBackend` with
  `backend=OpenShellSandbox(...)`. It is alpha and pins its own `deepagents` range, so install it in a separate
  environment if the versions conflict.
- [`langchain-nvidia-switchyard`](https://github.com/langchain-ai/langchain-nvidia/tree/main/libs/switchyard): an experimental `SwitchyardRoutingMiddleware` that
  routes each model call to a different target, for example a small Nemotron model for simple turns and a
  larger one for hard turns. Add it with `middleware=[SwitchyardRoutingMiddleware(router)]`.

See also:

- [Deep Agents documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents backends](https://docs.langchain.com/oss/python/deepagents/backends)
- [nemotron_tool_calling_agent.ipynb](https://github.com/langchain-ai/langchain-nvidia/blob/main/cookbook/nemotron_tool_calling_agent.ipynb) for a plain `create_agent` loop

## Related

- [Cookbook index](https://github.com/langchain-ai/langchain-nvidia/tree/main/cookbook)
- [`langchain-nvidia-ai-endpoints` README](https://github.com/langchain-ai/langchain-nvidia/blob/main/libs/ai-endpoints/README.md)
- [Model registry: current and deprecated model IDs](https://github.com/langchain-ai/langchain-nvidia/blob/main/libs/ai-endpoints/docs/model_registry.md)
- [NVIDIA NIM for LLMs documentation](https://docs.nvidia.com/nim/large-language-models/latest/introduction.html)